# Task 6 (Part 2): Generation for DPO-UF on AlpacaEval 300 & Cross-Track Evaluation (`align-generate-uf`)

In [ ]:
!nvidia-smi
import os, shutil, pathlib

input_root = pathlib.Path('/kaggle/input')
src_dir = next(input_root.rglob('submission_adapter.py')).parent
work_repo = pathlib.Path('/tmp/repo')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(src_dir, work_repo)
os.chdir(work_repo)
print('Found code dataset at:', src_dir)
print('COMMIT:', (work_repo / 'COMMIT.txt').read_text().strip())


In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers==4.57.6" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps
import torch, transformers
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
print('Transformers:', transformers.__version__)


In [ ]:
# 1. 3-prompt sanity check FIRST on DPO-UF (greedy, max_new_tokens=512, repetition_penalty=1.0)
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_uf/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/sanity_3_alpaca_dpo_uf.json \
  --generator_name qwen2.5-0.5b-dpo-ultrafeedback \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --limit 3 --batch_size 3

# 2. Full 300-prompt generation for DPO-UF on AlpacaEval 300 (greedy, repetition_penalty=1.0)
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_uf/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_dpo_uf.json \
  --generator_name qwen2.5-0.5b-dpo-ultrafeedback \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.0 --no_repeat_ngram_size 0 --batch_size 16

# 3. DPO-UF on AlpacaEval 300 with repetition_penalty=1.1 (for Rule 2 if SFT or DPO-UF > 10%)
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path outputs/dpo_uf/best \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_dpo_uf_rp11.json \
  --generator_name qwen2.5-0.5b-dpo-ultrafeedback-rp11 \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 16

# 4. Base Qwen2.5-0.5B on AlpacaEval 300 & HH-RLHF 300 with repetition_penalty=1.1
# (Required by Rule 2 for Base vs SFT because Base hit 30.33% and 25.33% > 10% under greedy rp=1.0)
!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path Qwen/Qwen2.5-0.5B \
  --eval_set_path eval_sets/alpaca_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/alpaca_base_rp11.json \
  --generator_name qwen2.5-0.5b-base-rp11 \
  --dataset_name alpaca_eval \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 16

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path Qwen/Qwen2.5-0.5B \
  --eval_set_path eval_sets/hh_rlhf_eval_300_seed42.json \
  --output_path /kaggle/working/outputs/generations/hh_base_rp11.json \
  --generator_name qwen2.5-0.5b-base-rp11 \
  --dataset_name hh_rlhf \
  --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --batch_size 16
